# One Frame Is Enough: RQ1 benchmark on Colab

**First time:** Runtime > Change runtime type > **GPU**, upload `eca_benchmark.zip` to `/content` (Files panel), Run all.
The code is copied once to `MyDrive/eca_project`; from then on that Drive folder is the code you run.

**Editing code:** Files panel (left) > `drive/MyDrive/eca_project/src/...` > double-click a `.py` file, edit, Ctrl+S.
Re-run the cell; `!python` cells always pick up the saved file. Changes persist in Drive across sessions.

**After a disconnect:** Run all. Data is re-downloaded to the local disk (~3 min); results in Drive resume from cache.
To change a system that already ran, delete its folders under `eca_results/rq1/dets/<system>` and `scores/<system>_t*`.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CODE = '/content/drive/MyDrive/eca_project'                    # your editable code
ZIP = '/content/eca_benchmark.zip'                             # only needed the first time
os.environ['ECA_DATA'] = '/content/data'                       # frames on fast local disk
OUT = '/content/drive/MyDrive/eca_results/rq1'
EGOBLUR = '/content/drive/MyDrive/egoblur/ego_blur_face.jit'   # skipped if missing
if os.path.exists(f'{CODE}/src'):
    print('Using the code already in Drive. To reseed from a new zip, delete MyDrive/eca_project first.')
else:
    assert os.path.exists(ZIP), 'Upload eca_benchmark.zip to /content first (Files panel)'
    !mkdir -p "$CODE" && unzip -q -o $ZIP -d "$CODE"
%cd $CODE
%load_ext autoreload
%autoreload 2

In [ ]:
# insightface pulls CPU onnxruntime, which shares a folder with onnxruntime-gpu: remove both, force the GPU build back
!pip install -q insightface deface
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q --force-reinstall --no-deps onnxruntime-gpu
# If Colab asks to restart the session: restart, then Run all again (this cell is safe to repeat).

In [ ]:
# Stops Run all if a model fell back to CPU. Results are still correct on CPU, only FPS is affected:
# comment this line out to continue anyway.
!python src/check_gpu.py

In [ ]:
!python tests/test_core.py

In [ ]:
!python src/loaders/chokepoint_download.py P2E_S1 P2L_S1   # ~1.4 GB to /content/data, the 12 sequences with GT

Sanity check: GT face boxes (derived from eye positions) should sit on the faces.

In [ ]:
import sys, cv2, matplotlib.pyplot as plt
sys.path.insert(0, 'src')
from loaders.chokepoint import load_gt, frame_path
seq = 'P2L_S1_C1.1'
gt = load_gt(seq)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, n in zip(axes, gt.frame.unique()[::len(gt.frame.unique()) // 4]):
    img = cv2.imread(str(frame_path(seq, n)))
    for r in gt[gt.frame == n].itertuples():
        cv2.rectangle(img, (int(r.x), int(r.y)), (int(r.x + r.w), int(r.y + r.h)), (0, 255, 0), 2)
    ax.imshow(img[..., ::-1]); ax.set_title(f'{seq} #{n}'); ax.axis('off')

In [ ]:
!python src/benchmark.py --systems yunet scrfd_500m --seqs P2E_S1_C2.2 P2L_S1_C1.2 --max-frames 400 --out /content/smoke   # ~1 min smoke test (needs 2 sessions)

In [ ]:
!python src/benchmark.py --egoblur $EGOBLUR --out $OUT

In [ ]:
import pandas as pd
pd.read_csv(f'{OUT}/results.csv').round(3)